In [ ]:
import os
from pyexpat.errors import messages
from typing import List

from openai import OpenAI
from  dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def get_completions(prompt, model):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return response


In [ ]:

get_completions("What is 1+1?", "gpt-4o-mini")

In [ ]:
import requests
import time
import os
from  dotenv import load_dotenv

_ = load_dotenv()

start = time.perf_counter()
llm_url = os.getenv("LLM_URL")
url = llm_url + "api/chat"
payload = {
    "model": "mistral",
    "messages": [
        {"role": "user", "content": "Waht is 1+1?"}
    ],
    "stream": False
}

response = requests.post(url, json=payload) #, timeout=120)
response.raise_for_status()
print(response.json()["message"]["content"])
# print("text:", response.text)
end = time.perf_counter()
print(end - start)

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("LLM_URL") + r"v1/",
    api_key="ollama"
)

# openai sdk compatible ollama call
def get_completion(prompt, model):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model= model,
        messages=messages,
        temperature=0
    )

    # return response.choices[0].message["content"]
    return response.choices[0].message.content



In [ ]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
style = """
American English \
in a calm and respectful tone
"""

In [ ]:
prompt = f"""
Translate the text \
that is delimited by the triple backticks \
into a style that is {style}.
text: ```{customer_email}```
"""
print(prompt)

In [ ]:
output = get_completion(prompt, "mistral")
print(output)

# Using Langchain

In [ ]:
# !pip install langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama

In [ ]:
chat = ChatOllama(
    model='mistral',
    temperature=0,
    base_url=os.getenv("LLM_URL"),
)

## Prompt Template




In [ ]:
template_string = """Translate the text \
that is delimited by the triple backticks \
into a style that is {style}.
text: ```{text}```
"""

In [ ]:
# !pip install langchain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [ ]:
prompt_template.messages[0].prompt

In [ ]:
prompt_template.messages[0].prompt.input_variables

In [ ]:
customer_style = """American English \
in a calm and respectful tone
"""

In [ ]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [ ]:
print(type(customer_messages))
print(type(customer_messages[0]))

In [ ]:
print(customer_messages[0])

In [ ]:
customer_response = chat.invoke(customer_messages)

In [ ]:
print(customer_response.content)

In [ ]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [ ]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [ ]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

In [ ]:
service_response = chat.invoke(service_messages)
print(service_response.content)

## Output Parsers

In [ ]:
{
    "gift": False,
    "delivery_days": 5,
    "price_value": "pretty affordable"
}

In [ ]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)

print(prompt_template)

In [ ]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOllama(
    model='mistral',
    temperature=0.0,
    base_url=os.getenv("LLM_URL"),
)
response = chat.invoke(messages)

print(response.content)

In [ ]:
type(response.content)

In [ ]:
response.content.get('gift')

### Parse the LLM output string into a Python dictionary using pydantic

In [ ]:
# from langchain.output_parsers import ResponseSchema
# from langchain.output_parsers import StructuredOutputParser
# Modern Alternative

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ReviewData(BaseModel):
    gift: str = Field(description="True or False")
    delivery_days: str = Field(description="How many days did it take for the product")
    price_value: List[str] = Field(description="Extract any sentences about the value or price")

structured_response = chat.with_structured_output(ReviewData)

result = structured_response.invoke(messages)

In [ ]:
print(result.gift)

In [ ]:
print(result.delivery_days)

In [ ]:
print(result.price_value)